In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

In [ ]:
df_part3.groupby("day")["vix_ln_return"].describe().mean()

In [ ]:
# # group data for lead_time:
# lead_cancel_data = df.groupby("month")["year"].describe()
# # use only lead_times wih more than 10 bookings for graph:
# lead_cancel_data_10 = lead_cancel_data.loc[lead_cancel_data["count"] >= 10]

#show figure:
plt.figure(figsize=(12, 8))
sns.regplot(x=df_part3.index, y=df_part3["sentiment"] )
plt.title("Effect of lead time on cancelation", fontsize=16)
plt.xlabel("Lead time", fontsize=16)
plt.ylabel("Cancelations [%]", fontsize=16)
# plt.xlim(0,365)
plt.show()

In [ ]:
import matplotlib.dates as mdates

# Dates
starting_date = dt.datetime(2018, 4, 7)
starting_date1 = mdates.date2num(starting_date)
trend_date = dt.datetime(2018, 6, 7)
trend_date1 = mdates.date2num(trend_date)

pointing_arrow = dt.datetime(2018, 2, 18)
pointing_arrow1 = mdates.date2num(pointing_arrow)

# Learn more Prophet tomorrow and plot the forecast for amazon.
fig = m.plot(df_part3["sentiment"])
ax1 = fig.add_subplot(111)
ax1.set_title("Amazon Stock Price Forecast", fontsize=16)
ax1.set_xlabel("Date", fontsize=12)
ax1.set_ylabel("Close Price", fontsize=12)

# Forecast initialization arrow
ax1.annotate('Forecast \n Initialization', xy=(pointing_arrow1, 1350), xytext=(starting_date1,1700),
            arrowprops=dict(facecolor='#ff7f50', shrink=0.1),
            )

# Trend emphasis arrow
ax1.annotate('Upward Trend', xy=(trend_date1, 1225), xytext=(trend_date1,950),
            arrowprops=dict(facecolor='#6cff6c', shrink=0.1),
            )

ax1.axhline(y=1260, color='b', linestyle='-')

plt.show()

In [ ]:
# Split data into train data(including validation data) and test data
split_ratio = 0.8

data = df_part3.set_index('day')
len_data = df_part3.shape[0]

train_org = df_part3.iloc[:int(len_data*split_ratio),]
test_org = df_part3.iloc[int(len_data*split_ratio):,]

print('train set',train_org.shape)
print('test set',test_org.shape)

In [ ]:
# start with simple scatterplot
sns.set(rc={"figure.figsize":(8, 4)})
sns.scatterplot(x='day',y='vix',data=df_part3)
plt.figure(figsize=(4, 4))
#plt.show()

# too bizarre when VKOSPI is small.
# print('Too bizarre when VKOSPI is small.')
# print('Instead of plotting every data, plot mean of VKOSPI within each Day_till_expiration.')
      
mean_by_dtexp = df_part3.groupby('day').mean()['vix']
plt.figure(figsize=(7,5))
# plt.ylabel('Mean of VIX')
# mean_by_dtexp_plot = mean_by_dtexp - 10  # for plotting purpose
# sns.barplot(x=mean_by_dtexp.index,y=mean_by_dtexp_plot.values,bottom=10, data=df_part3)
sns.barplot(x="day",y="vix", data=df_part3)

plt.show()

# reasonable to suspect that there aren't much data with very high day_till_expiration
print('It is reasonable to suspect that there aren\'t much data when day_till_expiration is high')
len_by_dtexp = df_part3['day'].value_counts()
plt.figure(figsize=(7,5))
plt.xlabel('day')
plt.ylabel('Frequency of day')
sns.barplot(x=len_by_dtexp.index,y=len_by_dtexp.values, data=df_part3)
plt.show()

print('The amount of data seems to be insufficient when Day_till_expiration is high')

In [ ]:
sns.lineplot(x="year", y="Value", data=df)


In [ ]:
from scipy import stats
def slice_and_anaylze(slice_at):
    # slice dataframe till the given parameter(slice_at)
    
    print(f'Plot from 0 to {slice_at}.')
    mean_by_dtexp = df_part3.groupby('day').mean()['vix']
    mean_by_dtexp = mean_by_dtexp[:slice_at]  # slice
    mean_by_dtexp = pd.DataFrame({'day':mean_by_dtexp.index,
                                  'Mean_of_VIX':mean_by_dtexp.values})  # Convert series to dataframe (for sns plot)
    
    # lmplot
    sns.lmplot(x='day',y='Mean_of_VIX',data=mean_by_dtexp)
    plt.show()

    # correlation test (pearson)
    corr_coef = stats.pearsonr(mean_by_dtexp['day'],mean_by_dtexp['Mean_of_VIX'])
    print('Pearson correlation test')
    print('='*50)
    print('correlation coefficient:',corr_coef[0])
    print('2-tailed p-value:', corr_coef[1])

In [ ]:
slice_and_anaylze(20)

In [ ]:

# 2017 onwards
last_year = df_part3.loc[df_part3['year'] > 2021]
g = last_year.groupby(["date"])
monthly_averages = g.aggregate({"open": np.mean, "high": np.mean, "low": np.mean, "close":np.mean})
monthly_averages.reset_index(level=0, inplace=True)

monthly_averages.dtypes


trace = go.Candlestick(x=monthly_averages['date'],
                       open=monthly_averages['open'].values.tolist(),
                       high=monthly_averages['high'].values.tolist(),
                       low=monthly_averages['low'].values.tolist(),
                       close=monthly_averages['close'].values.tolist(),
                      increasing=dict(line=dict(color= '#58FA58')),
                decreasing=dict(line=dict(color= '#FA5858')))


layout = {
    'title': 'Amazon Historical Price <br> <i>A closer look to the upward trend </i>',
    'xaxis': {'title': 'Date',
             'rangeslider': {'visible': False}},
    'yaxis': {'title': 'Stock Price (USD$)'}
}


data = [trace]

fig = go.Figure(data=data, layout=layout)
iplot(fig, filename='simple_ohlc')

In [ ]:
fig = go.Figure(go.Funnelarea(
    text =temp.sentiment,
    values = temp.text,
    title = {"position": "top center", "text": "Funnel-Chart of Sentiment Distribution"}
    ))
fig.show()